In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# 0. Criacao das Pastas de Saida
# -----------------------------------------------------------------------------
os.makedirs("../dados", exist_ok=True)
os.makedirs("../figuras", exist_ok=True)

# -----------------------------------------------------------------------------
# 1. Modelagem e Amostragem do Sinal Continuo de Temperatura (48 Horas)
# -----------------------------------------------------------------------------
T_total = 48 * 3600  # Duracao total: 48 horas em segundos
Ts = 300  # Intervalo de amostragem: 5 minutos (300 segundos)
fs = 1 / Ts  # Frequencia de amostragem (Hz)

t = np.arange(0, T_total, Ts)  # Vetor de tempo continuo discreto
n = np.arange(len(t))  # Indice das amostras

# Parametros do Fenomeno Fisico (Ciclo Circadiano + Harmonica)
T0 = 25.0  # Temperatura media (°C)
A1, f1 = 6.0, 1 / (24 * 3600)  # Componente diaria (24 h)
A2, f2 = 1.5, 1 / (12 * 3600)  # Harmonica secundaria (12 h)

# Sinal ideal continuo amostragem x[n]
x_ideal = (
    T0
    + A1 * np.sin(2 * np.pi * f1 * t - np.pi / 2)
    + A2 * np.sin(2 * np.pi * f2 * t)
)

# -----------------------------------------------------------------------------
# 2. Adicao de Ruido Gaussiano Aditivo (AWGN)
# -----------------------------------------------------------------------------
np.random.seed(42)  # Garantir reprodutibilidade
sigma_ruido = 0.8  # Desvio padrao do ruido (°C)
ruido = np.random.normal(0, sigma_ruido, size=len(n))
x_ruidoso = x_ideal + ruido

# -----------------------------------------------------------------------------
# 3. Quantizacao Uniforme Mid-Tread (4 bits e 8 bits)
# -----------------------------------------------------------------------------
Vmin, Vmax = 15.0, 35.0  # Faixa dinamica do sensor (°C)


def quantizar_sinal(x, n_bits, v_min, v_max):
    L = 2**n_bits
    delta = (v_max - v_min) / L
    x_sat = np.clip(x, v_min, v_max)
    indices = np.floor((x_sat - v_min) / delta)
    indices = np.clip(indices, 0, L - 1)
    x_q = v_min + (indices + 0.5) * delta
    return x_q, delta


x_q4, delta_4bit = quantizar_sinal(x_ruidoso, 4, Vmin, Vmax)
x_q8, delta_8bit = quantizar_sinal(x_ruidoso, 8, Vmin, Vmax)

# -----------------------------------------------------------------------------
# 4. Filtragem Digital LTI - Média Movel (M = 7 pontos)
# -----------------------------------------------------------------------------
M = 7
h_pb = np.ones(M) / M  # Resposta ao impulso do filtro passa-baixas
y_filtrado = np.convolve(x_ruidoso, h_pb, mode="same")

# -----------------------------------------------------------------------------
# 5. Exportacao dos Dados Gerados para mini-projeto/dados/
# -----------------------------------------------------------------------------
df_dados = pd.DataFrame(
    {
        "Amostra_n": n,
        "Tempo_s": t,
        "Tempo_h": t / 3600,
        "Temp_Ideal_C": x_ideal,
        "Temp_Ruidosa_C": x_ruidoso,
        "Temp_Quant_4bit_C": x_q4,
        "Temp_Quant_8bit_C": x_q8,
        "Temp_Filtrada_M7_C": y_filtrado,
    }
)

df_dados.to_csv("../dados/temperatura_completo.csv", index=False)
df_dados[["Tempo_h", "Temp_Ideal_C"]].to_csv(
    "../dados/temperatura_ideal.csv", index=False
)
df_dados[["Tempo_h", "Temp_Ruidosa_C"]].to_csv(
    "../dados/temperatura_ruidosa.csv", index=False
)
df_dados[["Tempo_h", "Temp_Quant_4bit_C"]].to_csv(
    "../dados/temperatura_quantizada_4bit.csv", index=False
)
df_dados[["Tempo_h", "Temp_Quant_8bit_C"]].to_csv(
    "../dados/temperatura_quantizada_8bit.csv", index=False
)
df_dados[["Tempo_h", "Temp_Filtrada_M7_C"]].to_csv(
    "../dados/temperatura_filtrada_m7.csv", index=False
)

# -----------------------------------------------------------------------------
# 6. Geração e Salvamento das Figuras em mini-projeto/figuras/
# -----------------------------------------------------------------------------
tempo_h = t / 3600

# Figura 1: Sinal Ideal vs Sinal Amostrado Ruidoso
plt.figure(figsize=(10, 4))
plt.plot(
    tempo_h,
    x_ideal,
    "k--",
    linewidth=1.5,
    label=r"Temperatura Ideal $x(t)$",
)
plt.plot(
    tempo_h,
    x_ruidoso,
    "r-",
    alpha=0.6,
    linewidth=0.8,
    label=r"Sinal Amostrado Ruidoso $x_r[n]$ ($\sigma=0.8$ °C)",
)
plt.title(
    "Amostragem do Sinal de Temperatura Ambiental (48h, $T_s=5$ min)"
)
plt.xlabel("Tempo (horas)")
plt.ylabel("Temperatura (°C)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(
    "../figuras/01_amostragem_sinal_temperatura.png", dpi=300
)
plt.close()

# Figura 2: Efeito da Quantização (4 bits vs 8 bits)
plt.figure(figsize=(10, 4))
plt.step(
    tempo_h[:72],
    x_q4[:72],
    "m-",
    where="post",
    label=rf"4 bits ($\Delta = {delta_4bit:.2f}$ °C)",
)
plt.step(
    tempo_h[:72],
    x_q8[:72],
    "c-",
    where="post",
    label=rf"8 bits ($\Delta = {delta_8bit:.4f}$ °C)",
)
plt.plot(
    tempo_h[:72],
    x_ideal[:72],
    "k--",
    alpha=0.8,
    label="Sinal Ideal Sem Ruído",
)
plt.title("Detalhe da Quantização Uniforme (Primeiras 6 Horas)")
plt.xlabel("Tempo (horas)")
plt.ylabel("Temperatura (°C)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig("../figuras/02_quantizacao_4bit_vs_8bit.png", dpi=300)
plt.close()

# Figura 3: Filtragem por Média Móvel
plt.figure(figsize=(10, 4))
plt.plot(
    tempo_h,
    x_ruidoso,
    color="gray",
    alpha=0.5,
    label="Sinal Ruidoso $x_r[n]$",
)
plt.plot(
    tempo_h,
    y_filtrado,
    "b-",
    linewidth=1.8,
    label="Sinal Filtrado $y[n]$ (Média Móvel $M=7$)",
)
plt.plot(
    tempo_h, x_ideal, "k--", linewidth=1.2, label="Sinal Ideal $x[n]$"
)
plt.title("Atenuação do Ruído via Filtro Passa-Baixas Média Móvel")
plt.xlabel("Tempo (horas)")
plt.ylabel("Temperatura (°C)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig("../figuras/03_filtragem_media_movel.png", dpi=300)
plt.close()

print(
    "Simulação executada com sucesso. Arquivos salvos em dados/ e figuras/."
)

Simulação executada com sucesso. Arquivos salvos em dados/ e figuras/.
